# Лабораторная работа №2  
## EDA и TSA на наборе данных Human Activity Recognition

**Цель работы:** провести разведочный анализ данных и анализ временных рядов на датасете HAR, содержащем сигналы акселерометра и гироскопа смартфона.

В работе выполняются:

1. загрузка и первичный осмотр данных;
2. подготовка табличного набора признаков;
3. анализ распределения классов активности;
4. визуализация сигналов акселерометра и гироскопа;
5. расчёт статистических признаков по временным окнам;
6. анализ автокорреляции, стационарности и частотной структуры сигнала;
7. небольшой пример прогнозирования одного сигнала;
8. итоговые выводы.

Источник данных: UCI HAR Dataset / Human Activity Recognition dataset.


In [ ]:
import os
import sys
import subprocess
import importlib
import warnings
from pathlib import Path
import zipfile
import urllib.request

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from scipy.fft import rfft, rfftfreq
from scipy import stats

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Загрузка датасета

В примерах одногруппников используется датасет Human Activity Recognition. Ниже сделана загрузка через `kagglehub`. Дополнительно добавлен запасной вариант загрузки напрямую из UCI, чтобы ноутбук был устойчивее.


In [ ]:
def ensure_import(package_name, import_name=None):
    """Импортирует пакет, а если его нет — устанавливает через pip."""
    import_name = import_name or package_name
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return importlib.import_module(import_name)


def find_har_dir(root):
    """Ищет папку UCI HAR Dataset по наличию ключевых файлов."""
    root = Path(root)
    for path in [root] + list(root.rglob("*")):
        if path.is_dir() and (path / "features.txt").exists() and (path / "activity_labels.txt").exists():
            return path
    return None


def download_dataset():
    """Загружает HAR dataset через KaggleHub или через UCI fallback."""
    try:
        kagglehub = ensure_import("kagglehub")
        dataset_path = kagglehub.dataset_download("erenaktas/human-activity-recognition")
        datadir = find_har_dir(dataset_path)
        if datadir is not None:
            return datadir
    except Exception as exc:
        print("KaggleHub не сработал, пробую загрузку из UCI:", exc)

    data_root = Path("data")
    data_root.mkdir(exist_ok=True)
    zip_path = data_root / "uci_har_dataset.zip"
    extract_dir = data_root / "uci_har"

    if not zip_path.exists():
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
        urllib.request.urlretrieve(url, zip_path)

    if not extract_dir.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(extract_dir)

    datadir = find_har_dir(extract_dir)
    if datadir is None:
        raise FileNotFoundError("Не удалось найти папку с UCI HAR Dataset")
    return datadir


datadir = download_dataset()
print("Папка с данными:", datadir)
print("Основные файлы:")
for file in sorted(datadir.glob("*.txt")):
    print("-", file.name)


## 2. Data Pipeline

В датасете есть два типа данных:

- `X_train.txt` / `X_test.txt` — готовые инженерные признаки по каждому окну наблюдения;
- папка `Inertial Signals` — исходные временные ряды по 128 точек для акселерометра и гироскопа.

Сначала загрузим инженерные признаки, метки активности и номера испытуемых.


In [ ]:
def make_unique(names):
    """Делает имена колонок уникальными. В HAR некоторые признаки повторяются."""
    seen = {}
    result = []
    for name in names:
        if name not in seen:
            seen[name] = 0
            result.append(name)
        else:
            seen[name] += 1
            result.append(f"{name}.{seen[name]}")
    return result


def load_txt_table(path):
    return pd.read_csv(path, sep=r"\s+", header=None)


features = pd.read_csv(datadir / "features.txt", sep=r"\s+", header=None, names=["feature_id", "feature_name"])
feature_names = make_unique(features["feature_name"].tolist())

activity_labels_df = pd.read_csv(datadir / "activity_labels.txt", sep=r"\s+", header=None, names=["activity_id", "activity_name"])
activity_map = dict(zip(activity_labels_df["activity_id"], activity_labels_df["activity_name"]))


def load_feature_split(split):
    X = load_txt_table(datadir / split / f"X_{split}.txt")
    X.columns = feature_names

    y = load_txt_table(datadir / split / f"y_{split}.txt").iloc[:, 0]
    subject = load_txt_table(datadir / split / f"subject_{split}.txt").iloc[:, 0]

    df = X.copy()
    df["activity_id"] = y.values
    df["activity_name"] = df["activity_id"].map(activity_map)
    df["subject"] = subject.values
    df["split"] = split
    return df


train_features = load_feature_split("train")
test_features = load_feature_split("test")
features_df = pd.concat([train_features, test_features], ignore_index=True)

print("Размер train:", train_features.shape)
print("Размер test:", test_features.shape)
print("Общий размер:", features_df.shape)
display(features_df.head())


## 3. Первичный EDA

Проверим структуру данных: пропуски, дубликаты, количество наблюдений по классам и испытуемым.


In [ ]:
eda_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "missing_values",
        "duplicates",
        "activities_count",
        "subjects_count"
    ],
    "value": [
        features_df.shape[0],
        features_df.shape[1],
        int(features_df.isna().sum().sum()),
        int(features_df.duplicated().sum()),
        features_df["activity_name"].nunique(),
        features_df["subject"].nunique()
    ]
})

display(eda_summary)
display(activity_labels_df)


In [ ]:
activity_counts = features_df["activity_name"].value_counts().rename_axis("activity").reset_index(name="count")
display(activity_counts)

plt.figure(figsize=(10, 5))
sns.barplot(data=activity_counts, x="activity", y="count")
plt.title("Количество наблюдений по видам активности")
plt.xlabel("Активность")
plt.ylabel("Количество окон")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


**Промежуточный вывод.** Если классы распределены примерно равномерно, значит, дальнейшие сравнения по активностям можно делать без сильного перекоса в одну категорию.


In [ ]:
subject_activity = pd.crosstab(features_df["subject"], features_df["activity_name"])
display(subject_activity.head())

plt.figure(figsize=(12, 6))
sns.heatmap(subject_activity, cmap="Blues")
plt.title("Распределение активностей по испытуемым")
plt.xlabel("Активность")
plt.ylabel("Испытуемый")
plt.tight_layout()
plt.show()


## 4. Анализ инженерных признаков

Посмотрим на готовые признаки, которые уже рассчитаны по временным окнам. Для наглядности возьмём несколько признаков, связанных со средним и стандартным отклонением ускорения.


In [ ]:
selected_columns = [
    "tBodyAcc-mean()-X",
    "tBodyAcc-mean()-Y",
    "tBodyAcc-mean()-Z",
    "tBodyAcc-std()-X",
    "tBodyAcc-std()-Y",
    "tBodyAcc-std()-Z",
    "tBodyGyro-mean()-X",
    "tBodyGyro-std()-X"
]
selected_columns = [col for col in selected_columns if col in features_df.columns]

summary_by_activity = features_df.groupby("activity_name")[selected_columns].agg(["mean", "std", "median"])
display(summary_by_activity)


In [ ]:
for col in selected_columns[:4]:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=features_df, x="activity_name", y=col)
    plt.title(f"Распределение признака {col} по активностям")
    plt.xlabel("Активность")
    plt.ylabel(col)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


## 5. PCA-визуализация

Сожмём 561 числовой признак до двух главных компонент. Это не модель классификации, а способ визуально проверить, разделяются ли активности в пространстве признаков.


In [ ]:
num_cols = features_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [col for col in num_cols if col not in ["activity_id", "subject"]]

sample_df = features_df.copy()
X_scaled = StandardScaler().fit_transform(sample_df[num_cols])

pca = PCA(n_components=2, random_state=42)
pca_values = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": pca_values[:, 0],
    "PC2": pca_values[:, 1],
    "activity_name": sample_df["activity_name"].values,
    "split": sample_df["split"].values
})

print("Доля объяснённой дисперсии двумя компонентами:", pca.explained_variance_ratio_.round(4))

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="activity_name", alpha=0.6, s=25)
plt.title("PCA-проекция наблюдений HAR dataset")
plt.xlabel("Главная компонента 1")
plt.ylabel("Главная компонента 2")
plt.legend(title="Активность", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 6. Загрузка исходных временных рядов

Теперь переходим к TSA: анализируем не готовые признаки, а сами сигналы из папки `Inertial Signals`.

Каждое наблюдение — это окно из 128 измерений. Частота дискретизации в UCI HAR Dataset — 50 Гц, поэтому одно окно соответствует примерно 2.56 секунды.


In [ ]:
SIGNAL_NAMES = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]


def signal_file_name(signal_name, split):
    return f"{signal_name}_{split}.txt"


def load_signal_matrix(split, signal_name):
    path = datadir / split / "Inertial Signals" / signal_file_name(signal_name, split)
    return pd.read_csv(path, sep=r"\s+", header=None).values


def load_inertial_split(split):
    arrays = [load_signal_matrix(split, signal_name) for signal_name in SIGNAL_NAMES]
    return np.stack(arrays, axis=2)


X_train_raw = load_inertial_split("train")
X_test_raw = load_inertial_split("test")
y_train = train_features["activity_id"].values
subject_train = train_features["subject"].values

print("X_train_raw shape:", X_train_raw.shape, "= наблюдения x время x сигнал")
print("X_test_raw shape:", X_test_raw.shape)


In [ ]:
def observation_to_df(X_raw, y, obs_index, fs=50):
    signal_df = pd.DataFrame(X_raw[obs_index], columns=SIGNAL_NAMES)
    signal_df["time_sec"] = np.arange(signal_df.shape[0]) / fs
    signal_df["activity_id"] = int(y[obs_index])
    signal_df["activity_name"] = activity_map[int(y[obs_index])]
    signal_df["obs_index"] = obs_index
    return signal_df


example_indices = (
    pd.DataFrame({"activity_id": y_train})
    .reset_index()
    .groupby("activity_id")["index"]
    .first()
    .tolist()
)

example_indices


## 7. Визуализация временных рядов

Для начала посмотрим, как отличаются сигналы акселерометра и гироскопа для разных типов активности.


In [ ]:
for obs_index in example_indices:
    signal_df = observation_to_df(X_train_raw, y_train, obs_index)
    activity_name = signal_df["activity_name"].iloc[0]

    plt.figure(figsize=(12, 5))
    plt.plot(signal_df["time_sec"], signal_df["total_acc_x"], label="total_acc_x")
    plt.plot(signal_df["time_sec"], signal_df["total_acc_y"], label="total_acc_y")
    plt.plot(signal_df["time_sec"], signal_df["total_acc_z"], label="total_acc_z")
    plt.title(f"Total acceleration для активности: {activity_name}")
    plt.xlabel("Время, сек")
    plt.ylabel("Амплитуда")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
selected_obs = example_indices[0]
selected_signal_df = observation_to_df(X_train_raw, y_train, selected_obs)
selected_activity = selected_signal_df["activity_name"].iloc[0]

signal_groups = {
    "Body acceleration": ["body_acc_x", "body_acc_y", "body_acc_z"],
    "Body gyroscope": ["body_gyro_x", "body_gyro_y", "body_gyro_z"],
    "Total acceleration": ["total_acc_x", "total_acc_y", "total_acc_z"]
}

for group_name, cols in signal_groups.items():
    plt.figure(figsize=(12, 5))
    for col in cols:
        plt.plot(selected_signal_df["time_sec"], selected_signal_df[col], label=col)
    plt.title(f"{group_name}. Наблюдение {selected_obs}, активность: {selected_activity}")
    plt.xlabel("Время, сек")
    plt.ylabel("Амплитуда")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 8. Расчёт статистических признаков по окнам

Для каждого окна временного ряда посчитаем простые статистики: среднее, стандартное отклонение, минимум, максимум, медиану, межквартильный размах и энергию сигнала.

Такие признаки часто используются как базовый этап обработки временных рядов с датчиков.


In [ ]:
def iqr(values):
    return np.percentile(values, 75) - np.percentile(values, 25)


def build_window_stats(X_raw, y, subjects, split):
    rows = []
    for obs_idx in range(X_raw.shape[0]):
        row = {
            "obs_index": obs_idx,
            "activity_id": int(y[obs_idx]),
            "activity_name": activity_map[int(y[obs_idx])],
            "subject": int(subjects[obs_idx]),
            "split": split
        }
        for signal_idx, signal_name in enumerate(SIGNAL_NAMES):
            values = X_raw[obs_idx, :, signal_idx]
            row[f"{signal_name}_mean"] = float(np.mean(values))
            row[f"{signal_name}_std"] = float(np.std(values))
            row[f"{signal_name}_min"] = float(np.min(values))
            row[f"{signal_name}_max"] = float(np.max(values))
            row[f"{signal_name}_median"] = float(np.median(values))
            row[f"{signal_name}_iqr"] = float(iqr(values))
            row[f"{signal_name}_energy"] = float(np.sum(values ** 2) / len(values))
        rows.append(row)
    return pd.DataFrame(rows)


stats_df = build_window_stats(X_train_raw, y_train, subject_train, "train")
print("Размер таблицы статистических признаков:", stats_df.shape)
display(stats_df.head())


In [ ]:
energy_cols = [col for col in stats_df.columns if col.endswith("_energy")]
energy_summary = stats_df.groupby("activity_name")[energy_cols].mean().round(4)
display(energy_summary)

plt.figure(figsize=(11, 5))
sns.boxplot(data=stats_df, x="activity_name", y="total_acc_x_energy")
plt.title("Энергия сигнала total_acc_x по активностям")
plt.xlabel("Активность")
plt.ylabel("Energy")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


**Промежуточный вывод.** Динамические активности обычно дают более высокую энергию и разброс сигнала, а статические активности — более спокойные временные ряды. Это хорошо видно на статистиках по окнам.


In [ ]:
corr_cols = [
    "body_acc_x_std", "body_acc_y_std", "body_acc_z_std",
    "body_gyro_x_std", "body_gyro_y_std", "body_gyro_z_std",
    "total_acc_x_energy", "total_acc_y_energy", "total_acc_z_energy"
]

plt.figure(figsize=(10, 8))
sns.heatmap(stats_df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляция статистических признаков временных рядов")
plt.tight_layout()
plt.show()


## 9. TSA: скользящее среднее и разброс

Выберем один сигнал и посмотрим его сглаженную динамику. Скользящее среднее помогает увидеть локальный тренд внутри короткого окна наблюдения.


In [ ]:
fs = 50
signal_name = "total_acc_x"
signal_idx = SIGNAL_NAMES.index(signal_name)
series = pd.Series(X_train_raw[selected_obs, :, signal_idx], name=signal_name)
time_sec = np.arange(len(series)) / fs

rolling_mean = series.rolling(window=10, min_periods=1).mean()
rolling_std = series.rolling(window=10, min_periods=1).std().fillna(0)

plt.figure(figsize=(12, 5))
plt.plot(time_sec, series, label="исходный сигнал", alpha=0.65)
plt.plot(time_sec, rolling_mean, label="скользящее среднее, окно=10")
plt.fill_between(time_sec, rolling_mean - rolling_std, rolling_mean + rolling_std, alpha=0.2, label="± rolling std")
plt.title(f"Скользящее среднее для {signal_name}, активность: {selected_activity}")
plt.xlabel("Время, сек")
plt.ylabel("Амплитуда")
plt.legend()
plt.tight_layout()
plt.show()


## 10. TSA: автокорреляция и частичная автокорреляция

ACF показывает связь ряда с его лаговыми значениями. PACF показывает связь с лагом после исключения влияния промежуточных лагов.


In [ ]:
plot_acf(series, lags=40)
plt.title(f"ACF для {signal_name}")
plt.tight_layout()
plt.show()

plot_pacf(series, lags=40, method="ywm")
plt.title(f"PACF для {signal_name}")
plt.tight_layout()
plt.show()


In [ ]:
ljung_result = acorr_ljungbox(series, lags=[10, 20], return_df=True)
display(ljung_result)

print("Если p-value маленькое, то ряд не похож на белый шум: в нём есть автокорреляционная структура.")


## 11. TSA: проверка стационарности

Используем ADF-тест. Нулевая гипотеза ADF: ряд имеет единичный корень, то есть является нестационарным.


In [ ]:
def adf_report(values):
    result = adfuller(values)
    report = pd.DataFrame({
        "metric": ["ADF statistic", "p-value", "used_lags", "n_obs"],
        "value": [result[0], result[1], result[2], result[3]]
    })
    critical = pd.DataFrame({
        "level": list(result[4].keys()),
        "critical_value": list(result[4].values())
    })
    return report, critical


adf_main, adf_critical = adf_report(series)
display(adf_main)
display(adf_critical)

p_value = float(adf_main.loc[adf_main["metric"] == "p-value", "value"].iloc[0])
if p_value < 0.05:
    print("Вывод: p-value < 0.05, ряд можно считать стационарным на 5% уровне значимости.")
else:
    print("Вывод: p-value >= 0.05, нет оснований уверенно считать ряд стационарным.")


## 12. TSA: частотный анализ через FFT

Быстрое преобразование Фурье позволяет посмотреть, какие частоты сильнее всего выражены в сигнале.


In [ ]:
centered = series.values - series.values.mean()
yf = np.abs(rfft(centered))
xf = rfftfreq(len(centered), 1 / fs)

non_zero = xf > 0
dominant_frequency = xf[non_zero][np.argmax(yf[non_zero])]

plt.figure(figsize=(12, 5))
plt.plot(xf[non_zero], yf[non_zero])
plt.title(f"Амплитудный спектр FFT для {signal_name}")
plt.xlabel("Частота, Гц")
plt.ylabel("Амплитуда")
plt.tight_layout()
plt.show()

print(f"Доминирующая частота: {dominant_frequency:.3f} Гц")


## 13. Мини-пример прогнозирования одного сигнала

Для полноты TSA попробуем построить простую SARIMAX/ARMA-модель на одном коротком сигнале. Это не промышленный прогноз, а демонстрация подхода к моделированию временного ряда.


In [ ]:
train_size = 96
train_series = series.iloc[:train_size]
test_series = series.iloc[train_size:]

try:
    model = SARIMAX(
        train_series,
        order=(2, 0, 1),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    fitted = model.fit(disp=False)
    forecast = fitted.forecast(steps=len(test_series))

    rmse = mean_squared_error(test_series, forecast, squared=False)
    r2 = r2_score(test_series, forecast)

    plt.figure(figsize=(12, 5))
    plt.plot(time_sec[:train_size], train_series, label="train")
    plt.plot(time_sec[train_size:], test_series, label="test")
    plt.plot(time_sec[train_size:], forecast, label="forecast")
    plt.title(f"Прогноз SARIMAX для {signal_name}")
    plt.xlabel("Время, сек")
    plt.ylabel("Амплитуда")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"RMSE: {rmse:.4f}")
    print(f"R2: {r2:.4f}")
except Exception as exc:
    print("SARIMAX не удалось обучить на выбранном коротком ряде:", exc)


## 14. Сохранение подготовленных данных

Сохраним статистические признаки по окнам в CSV. Их можно использовать для дальнейшего моделирования.


In [ ]:
stats_df.to_csv("har_window_statistics.csv", index=False)
features_df.to_csv("har_engineered_features.csv", index=False)

print("Файлы сохранены:")
print("- har_window_statistics.csv")
print("- har_engineered_features.csv")


## Итоговые выводы

1. Данные HAR состоят из нескольких уровней: готовых инженерных признаков и исходных временных рядов акселерометра/гироскопа.
2. Пропусков в загруженной таблице нет, структура данных пригодна для анализа.
3. Классы активности представлены достаточно равномерно, поэтому сравнение активностей не искажается сильным дисбалансом.
4. По boxplot-графикам видно, что признаки ускорения и гироскопа различаются между статическими и динамическими активностями.
5. PCA показывает, что часть активностей визуально отделяется уже в пространстве двух главных компонент, хотя полного разделения двумя компонентами недостаточно.
6. Временные ряды имеют выраженную структуру: скользящее среднее, ACF/PACF и Ljung-Box показывают наличие зависимости между соседними значениями.
7. ADF-тест позволяет формально проверить стационарность выбранного сигнала.
8. FFT помогает выделить доминирующие частоты внутри окна наблюдения.
9. Простая SARIMAX-модель применима как демонстрация прогнозирования, но для HAR-задачи обычно важнее классификация активности, а не долгосрочный прогноз короткого сигнала.

**Общий вывод:** EDA и TSA позволяют понять структуру HAR-датасета, различия между активностями и свойства сенсорных временных рядов до построения ML-моделей.
